# 第 13 课：CTC 解码——从 Greedy 到 Prefix Beam Search

目标：理解为什么“每帧第一名”不等于“文本第一名”，并逐行实现 Prefix Beam Search。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | CTC 核心 |
| 建议投入 | 4～6 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 12 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | greedy path、prefix beam、p_blank/p_nonblank |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：greedy path、prefix beam、p_blank/p_nonblank。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_root()
BLANK = "∅"
plt.rcParams["figure.figsize"] = (11, 4)
print("项目根目录:", ROOT)

from collections import defaultdict
from itertools import product
from ipywidgets import interact, IntSlider

symbols=[BLANK,"A","B"]
probs=np.array([[.40,.35,.40],[.35,.40,.10],[.25,.25,.50]])

def collapse(path):
    out=[];prev=None
    for x in path:
        if x!=prev and x!=BLANK: out.append(x)
        prev=x
    return "".join(out)

项目根目录: G:\learn_asr


## 1. Greedy decoding

In [2]:
path=[symbols[i] for i in probs.argmax(0)]
print("greedy path:",path,"text:",collapse(path),"path probability:",np.prod(probs.max(0)))

greedy path: ['∅', 'A', 'B'] text: AB path probability: 0.08000000000000002


## 2. 用穷举计算每个文本的精确概率

In [3]:
totals=defaultdict(float)
for path in product(symbols,repeat=probs.shape[1]):
    p=np.prod([probs[symbols.index(s),t] for t,s in enumerate(path)])
    totals[collapse(path)]+=p
for text,p in sorted(totals.items(),key=lambda x:x[1],reverse=True): print(repr(text),f"{p:.5f}")

'AB' 0.29000
'B' 0.25125
'A' 0.21300
'BA' 0.07500
'' 0.05600
'BAB' 0.05000
'BB' 0.04375
'AA' 0.01225
'ABA' 0.00875


## 3. Prefix Beam Search 的两个概率

对每个前缀保存：

- `p_b(prefix)`：以 blank 结尾的路径概率和
- `p_nb(prefix)`：以非 blank 结尾的路径概率和

必须分开保存，才能正确处理 `A A` 与 `A blank A`。

In [4]:
def prefix_beam_search(P, labels, beam_size=5):
    beam={"":(1.0,0.0)}
    history=[]
    for t in range(P.shape[1]):
        nxt=defaultdict(lambda:[0.0,0.0])
        for prefix,(pb,pnb) in beam.items():
            for i,c in enumerate(labels):
                p=P[i,t]
                if c==BLANK:
                    nxt[prefix][0]+=(pb+pnb)*p
                elif prefix and c==prefix[-1]:
                    nxt[prefix][1]+=pnb*p
                    nxt[prefix+c][1]+=pb*p
                else:
                    nxt[prefix+c][1]+=(pb+pnb)*p
        beam=dict(sorted(nxt.items(),key=lambda kv:sum(kv[1]),reverse=True)[:beam_size])
        history.append(beam)
    return beam,history

beam,history=prefix_beam_search(probs,symbols,beam_size=20)
for prefix,(pb,pnb) in sorted(beam.items(),key=lambda x:sum(x[1]),reverse=True)[:8]:
    print(repr(prefix),f"total={pb+pnb:.5f}",f"blank={pb:.5f}",f"nonblank={pnb:.5f}")

'AB' total=0.29000 blank=0.03500 nonblank=0.25500
'B' total=0.25125 blank=0.10000 nonblank=0.15125
'A' total=0.21300 blank=0.16900 nonblank=0.04400
'BA' total=0.07500 blank=0.04000 nonblank=0.03500
'' total=0.05600 blank=0.05600 nonblank=0.00000
'BAB' total=0.05000 blank=0.00000 nonblank=0.05000
'BB' total=0.04375 blank=0.00000 nonblank=0.04375
'AA' total=0.01225 blank=0.00000 nonblank=0.01225


## 4. 交互观察 beam 如何随时间变化

In [5]:
@interact(t=IntSlider(min=1,max=probs.shape[1],value=1,description="已处理帧"))
def show_beam(t=1):
    items=sorted(history[t-1].items(),key=lambda x:sum(x[1]),reverse=True)
    names=[k or "<empty>" for k,_ in items]; vals=[sum(v) for _,v in items]
    plt.barh(names[::-1],vals[::-1]); plt.xlabel("Prefix probability");plt.title(f"Beam after frame {t}");plt.show()

interactive(children=(IntSlider(value=1, description='已处理帧', max=3, min=1), Output()), _dom_classes=('widget-i…

## 5. Beam size 是速度与准确率旋钮

beam 太小会过早剪掉后来可能翻盘的前缀；beam 太大增加 CPU、内存和延迟。工业系统还常用相对阈值 beam pruning。

In [6]:
for k in [1,2,3,5,20]:
    b,_=prefix_beam_search(probs,symbols,k)
    best=max(b.items(),key=lambda x:sum(x[1]))
    print("beam",k,"->",repr(best[0]),sum(best[1]))

beam 1 -> 'A' 0.08000000000000002
beam 2 -> 'A' 0.21300000000000005
beam 3 -> 'B' 0.25125
beam 5 -> 'AB' 0.29
beam 20 -> 'AB' 0.29


## 本课测试

1. Greedy 搜索的是路径还是文本？
2. 为什么 prefix 要分 `p_b` 和 `p_nb`？
3. beam size 越大是否一定更适合实时系统？
4. blank 会不会出现在最终前缀中？
5. 后面接语言模型时，应在什么时机增加 token 的 LM 分数？

<details><summary>展开参考答案</summary>

1. 单条逐帧路径。2. 为了正确处理重复 token。3. 不一定，会增加计算和延迟。4. 不会，blank 只改变概率状态。5. 只有当前 token 真正扩展了输出前缀时。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 13 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `greedy path`、`prefix beam`、`p_blank/p_nonblank`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**beam size=1 且存在多路径累加翻盘**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**从空白实现 prefix beam 并与穷举对照**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**为第 20 课接入语言模型保留扩展点**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：greedy path、prefix beam、p_blank/p_nonblank。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 greedy path、prefix beam、p_blank/p_nonblank。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
